In [2]:
# Setup the Jupyter version of Dash
from jupyter_dash import JupyterDash

# Configure the necessary Python module imports for dashboard components
import dash_leaflet as dl
from dash import dcc, html, dash_table
import plotly.express as px
from dash.dependencies import Input, Output
import base64
import os
import pandas as pd

# Import your CRUD Python module class
from CRUD_Python_Module import AnimalShelter

JupyterDash.infer_jupyter_proxy_config()


###########################
# Data Manipulation / Model
###########################

username = "aacuser"
password = "aacpassword"

# Connect to database via CRUD Module
db = AnimalShelter(username, password)


def load_data(query):
    """Load data from MongoDB and return a cleaned DataFrame."""
    df = pd.DataFrame.from_records(db.read(query))

    if not df.empty and "_id" in df.columns:
        df.drop(columns=["_id"], inplace=True)

    return df


# Initial unfiltered data
df = load_data({})


#########################
# Dashboard Layout / View
#########################

app = JupyterDash(__name__)

# Add Grazioso Salvare logo
image_filename = "Grazioso Salvare Logo.png"
encoded_image = None

if os.path.exists(image_filename):
    with open(image_filename, "rb") as image_file:
        encoded_image = base64.b64encode(image_file.read()).decode()

app.layout = html.Div([
    html.Div([
        html.Div([
            html.Img(
                src=f"data:image/png;base64,{encoded_image}",
                style={"height": "120px", "display": "block", "margin": "auto"}
            ) if encoded_image else html.Div("Logo file not found")
        ], style={"textAlign": "center"}),

        html.H1(
            "Grazioso Salvare Rescue Dog Dashboard",
            style={"textAlign": "center"}
        ),
        html.H3(
            "Brandon Shank",
            style={"textAlign": "center", "color": "#444"}
        )
    ]),

    html.Hr(),

    html.Div([
        html.Label(
            "Select Rescue Filter:",
            style={"fontWeight": "bold"}
        ),
        dcc.Dropdown(
            id="filter-type",
            options=[
                {"label": "Reset / Show All", "value": "reset"},
                {"label": "Water Rescue", "value": "water"},
                {"label": "Mountain or Wilderness Rescue", "value": "mountain"},
                {"label": "Disaster or Individual Tracking", "value": "disaster"}
            ],
            value="reset",
            clearable=False,
            style={"width": "300px", "marginBottom": "20px"}
        )
    ], style={
        "marginBottom": "40px",
        "padding": "10px",
        "backgroundColor": "#f9f9f9",
        "borderRadius": "8px",
        "width": "320px"
    }),

    html.Hr(),

    dash_table.DataTable(
        id="datatable-id",
        columns=[{"name": i, "id": i, "deletable": False, "selectable": True} for i in df.columns],
        data=df.to_dict("records"),
        page_size=10,
        sort_action="native",
        filter_action="native",
        row_selectable="single",
        selected_rows=[0] if not df.empty else [],
        style_table={
            "overflowX": "auto"
        },
        style_cell={
            "textAlign": "left",
            "minWidth": "120px",
            "width": "120px",
            "maxWidth": "180px",
            "whiteSpace": "normal"
        },
        style_header={
            "backgroundColor": "rgb(230, 230, 230)",
            "fontWeight": "bold"
        }
    ),

    html.Br(),
    html.Hr(),

    html.Div(
        style={
            "display": "flex",
            "gap": "20px",
            "alignItems": "flex-start"
        },
        children=[
            html.Div(
                dcc.Loading(
                    type="default",
                    children=html.Div(id="graph-id")
                ),
                style={"width": "50%"}
            ),
            html.Div(
                dcc.Loading(
                    type="default",
                    children=html.Div(id="map-id")
                ),
                style={"width": "50%"}
            )
        ]
    )
])


#############################################
# Interaction Between Components / Controller
#############################################

@app.callback(
    Output("datatable-id", "data"),
    [Input("filter-type", "value")]
)
def update_dashboard(filter_type):
    """Filter interactive data table with MongoDB queries."""

    if filter_type == "water":
        query = {
            "animal_type": "Dog",
            "breed": {
                "$in": [
                    "Labrador Retriever Mix",
                    "Chesapeake Bay Retriever",
                    "Newfoundland"
                ]
            },
            "sex_upon_outcome": "Intact Female",
            "age_upon_outcome_in_weeks": {"$gte": 26, "$lte": 156}
        }

    elif filter_type == "mountain":
        query = {
            "animal_type": "Dog",
            "breed": {
                "$in": [
                    "German Shepherd",
                    "Alaskan Malamute",
                    "Old English Sheepdog",
                    "Siberian Husky",
                    "Rottweiler"
                ]
            },
            "sex_upon_outcome": "Intact Male",
            "age_upon_outcome_in_weeks": {"$gte": 26, "$lte": 156}
        }

    elif filter_type == "disaster":
        query = {
            "animal_type": "Dog",
            "breed": {
                "$in": [
                    "Doberman Pinscher",
                    "German Shepherd",
                    "Golden Retriever",
                    "Bloodhound",
                    "Rottweiler"
                ]
            },
            "sex_upon_outcome": "Intact Male",
            "age_upon_outcome_in_weeks": {"$gte": 20, "$lte": 300}
        }

    else:
        query = {}

    dff = load_data(query)
    return dff.to_dict("records")


@app.callback(
    Output("graph-id", "children"),
    [Input("datatable-id", "derived_virtual_data")]
)
def update_graphs(viewData):
    """Display top 10 breed chart based on filtered table data."""
    if viewData is None or len(viewData) == 0:
        return [html.H4("No data to display")]

    dff = pd.DataFrame(viewData)

    if "breed" not in dff.columns:
        return [html.H4("Breed column not found")]

    top_breeds = dff["breed"].value_counts().nlargest(10).reset_index()
    top_breeds.columns = ["breed", "count"]
    top_breeds = top_breeds.sort_values(by="count", ascending=False)

    fig = px.bar(
        top_breeds,
        x="breed",
        y="count",
        title="Top 10 Breeds in Current Filter"
    )

    fig.update_layout(
        xaxis_title="Breed",
        yaxis_title="Count",
        xaxis_tickangle=-45
    )

    return [dcc.Graph(figure=fig)]


@app.callback(
    Output("datatable-id", "style_data_conditional"),
    [Input("datatable-id", "selected_columns")]
)
def update_styles(selected_columns):
    """Highlight selected columns in the data table."""
    if selected_columns is None:
        return []

    return [{
        "if": {"column_id": i},
        "background_color": "#D2F3FF"
    } for i in selected_columns]


@app.callback(
    Output("map-id", "children"),
    [Input("datatable-id", "derived_virtual_data"),
     Input("datatable-id", "derived_virtual_selected_rows")]
)
def update_map(viewData, selected_rows):
    """Update geo-location map for the selected data entry."""
    if viewData is None or len(viewData) == 0:
        return [html.H4("No map data available")]

    dff = pd.DataFrame(viewData)

    required_columns = ["location_lat", "location_long", "breed", "name"]
    for col in required_columns:
        if col not in dff.columns:
            return [html.H4(f"Required column missing: {col}")]

    row = 0
    if selected_rows and len(selected_rows) > 0:
        row = selected_rows[0]

    if row >= len(dff):
        row = 0

    lat = float(dff.iloc[row]["location_lat"])
    lon = float(dff.iloc[row]["location_long"])
    breed = str(dff.iloc[row]["breed"])
    animal_name = str(dff.iloc[row]["name"])

    return [
        html.Div([
            html.H3(
                f"Selected Animal: {animal_name}",
                style={"color": "#1F4E79", "marginBottom": "5px"}
            ),
            html.H4(
                f"Breed: {breed}",
                style={"marginBottom": "5px"}
            ),
            html.P(
                f"Lat: {lat:.4f}, Lon: {lon:.4f}",
                style={"fontSize": "12px", "color": "#555", "marginBottom": "10px"}
            ),
            dl.Map(
                center=[lat, lon],
                zoom=15,
                style={"width": "100%", "height": "500px"},
                children=[
                    dl.TileLayer(),
                    dl.Marker(
                        position=[lat, lon],
                        children=[
                            dl.Tooltip(f"{animal_name} - {breed}")
                        ]
                    )
                ]
            )
        ])
    ]


# Run app and display result in JupyterLab mode
app.run_server()

Dash app running on https://athletetunnel-beginfortune-3000.codio.io/proxy/8050/
